<a href="https://colab.research.google.com/github/gilIolgenblum/CrowdingModeWorkshop/blob/main/tutorials/02_fit_experimental_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fit to Experimental Data

## Goal
In this tutorial, you will learn how to fit the soft interaction parameters ($\varepsilon$ and $\varepsilon_{TS}$) to experimental protein stability data using the crowding model.

The fitting procedure iteratively calls `solve_equil()` while adjusting the model parameters to minimize the residuals between the calculated and experimental data:
![Fit Data Workflow](assets/fit_data_flowchart.svg)

We will use experimental data for the AQ16 peptide in the presence of Trehalose (see *Olgenblum, Carmon, and Harries (2023)*).

## Setup
First, we import the package, as well as `pandas` for data handling and `matplotlib` for plotting.

In [ ]:
import os
import sys

# Check if we are running in Google Colab
if 'google.colab' in str(get_ipython()):
    # Clone the repository to get the data files
    !git clone https://github.com/gilIolgenblum/CrowdingModeWorkshop.git
    # Change the working directory to the repository root
    os.chdir('/content/CrowdingModeWorkshop')
    # Install the package dependencies
    !pip install -e .

    # TELL PYTHON WHERE THE SOURCE FOLDER IS
    sys.path.append('/content/CrowdingModeWorkshop/src')

In [ ]:
import crowding as cr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

print(f"Module imported successfully: {cr.__name__}")

## Load Experimental Data
The package includes sample datasets in `app/sample_data/`. Let's load the Trehalose dataset.

This dataset has the following columns:
- `concentration`: The concentration of Trehalose in **Molal** ($m$).
- `dG`: $\Delta\Delta G^0$ in **kJ/mol**.
- `dH`: $\Delta\Delta H^0$ in **kJ/mol**.
- `TdS`: $T\Delta\Delta S^0$ in **kJ/mol**.

In [ ]:
# Locate the sample data path
ROOT = Path.cwd()
while not (ROOT / "app").exists() and ROOT.parent != ROOT:
    ROOT = ROOT.parent

sample_data_path = ROOT / "app" / "sample_data" / "aq16_trehalose_binary_format1.csv"

# Load the data
df = pd.read_csv(sample_data_path)
display(df.head(10))

## Data Preparation
The package natively handles molal concentrations during fitting, so we don't need to manually convert the concentration to volume fraction ($\phi$), we just need to tell the model `concentration_type='molal'` later.
The energies are already in **kJ/mol**, which is exactly what the model expects.

In [ ]:
# Drop NaNs to create specific arrays for each fit
# For dG (eps fitting)
df_dG = df.dropna(subset=["concentration", "dG"])
exp_conc_G = df_dG["concentration"].values
exp_dG = df_dG["dG"].values

# For dH and TdS (epsTS fitting)
df_H_S = df.dropna(subset=["concentration", "dH", "TdS"])
exp_conc_HS = df_H_S["concentration"].values
exp_dH = df_H_S["dH"].values
exp_TdS = df_H_S["TdS"].values

## Define Model Parameters
To construct the model, we need the properties of AQ16 (Protein) and Trehalose (Cosolute).

For AQ16, the measured SASA is $242.6 \ \mathring{A}^2$.
For Trehalose, the physical parameters are $\nu=11.7$, $\chi=0.433$, and $\chi_{TS}=-1.12$.

In [ ]:
# Define Protein
AQ16 = cr.Protein(SASA=242.6)

# Define Trehalose Cosolute
trehalose = cr.Cosolute(
    nu=11.70, chi=0.433, chiTS=-1.120
)

# Initialize the model (we start with eps=0.0 and epsTS=0.0)
# We set a large enough phiC_max to ensure our experimental range is covered.
# Molal concentrations of ~1.3 m correspond to roughly phi=0.22.
model = cr.CrowdingModel(
    protein=AQ16,
    cosolute=trehalose,
    eps=0.0,
    epsTS=0.0,
    phiC_max=0.22,
    dphiC=0.0001,
    T=298.15
)

## Fit $\varepsilon$ from $\Delta\Delta G^0$
First, we fit the primary soft interaction parameter $\varepsilon$ using the $\Delta\Delta G^0$ data.
We use the `fit_eps` method and explicitly pass `concentration_type="molal"`.

In [ ]:
# Fit eps
model.fit_eps(
    exp_conc=???,
    exp_ddG=???,
    concentration_type='???',
    progress_bar=True
)

print(f"Fitted eps: {model.eps:.4f}")

## Fit $\varepsilon_{TS}$ from $\Delta\Delta H^0$ and $T\Delta\Delta S^0$
Next, now that $\varepsilon$ is fixed, we can fit the entropic component of the soft interactions ($\varepsilon_{TS}$) using the enthalpic and entropic data with the `fit_epsTS` method.

In [ ]:
# Fit epsTS
model.fit_epsTS(
    exp_conc=???,
    exp_ddH=???,
    exp_TddS=???,
    concentration_type='???',
    progress_bar=True
)

print(f"Fitted epsTS: {model.epsTS:.4f}")

print('----------------------\n',model)

## Visualizing the Fit
We can use the built-in `Plotter` to see how well the model curve matches the experimental scatter points. We pass the experimental data and the `concentration_type` directly to the `plot_results` function!

In [ ]:
plotter = cr.Plotter(model)

# The plotter will overlay the experimental scatter points over the solved model lines
fig = plotter.plot_results(
    concentration_type='molal',
    exp_conc=???,
    exp_ddG=???,
    exp_ddH=???,
    exp_TddS=???
)
plt.show()

## Exporting the Model Results
Now that the model is fully fitted and solved across the concentration grid, we can convert it to a DataFrame and export it.

In [ ]:
# Convert model grid results to a dataframe
model.to_pandas()
results = model.results
display(results.head())

# Export the dataframe
# results.to_csv("aq16_trehalose_fitted_model.csv", index=False)

# You can also access the fitted parameters directly for export:
params_dict = {
    "eps": model.eps,
    "epsTS": model.epsTS
}
print(f"Final Parameters: {params_dict}")